In [1]:
import os
import pandas as pd

In [2]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [3]:
data_path = 'data_antoni'
os.makedirs(data_path, exist_ok=True)

pdf_filenames = [filename for filename in os.listdir(data_path) if filename.endswith('.pdf')]
pdf_filenames

['20250415_NOTIFmODIF_firmado.pdf',
 'Estatutos_SUM_Alberic_castellano_.pdf',
 'Estatutos_SUM_Alberic_castellano.pdf',
 'CIF_SUMA_Actualitzat.pdf',
 '20250415_ESTATUTOS_DILIGENCIA.pdf',
 'MANUAL DE DIRECTIVAS - def (v. enero 2024).pdf',
 '03_2_Listado_Bienes.pdf']

In [4]:
from langchain_community.document_loaders import PyPDFLoader


all_docs = []

for filename in pdf_filenames:
    print(f"Loading {filename}")
    loader = PyPDFLoader(os.path.join(data_path, filename),
                         mode='single')

    all_docs.extend(loader.load())

Loading 20250415_NOTIFmODIF_firmado.pdf
Loading Estatutos_SUM_Alberic_castellano_.pdf
Loading Estatutos_SUM_Alberic_castellano.pdf
Loading CIF_SUMA_Actualitzat.pdf
Loading 20250415_ESTATUTOS_DILIGENCIA.pdf
Loading MANUAL DE DIRECTIVAS - def (v. enero 2024).pdf
Loading 03_2_Listado_Bienes.pdf


In [5]:
len(all_docs)

7

SEMANTIC CHUNK


In [6]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker

model_name = "intfloat/e5-large-v2" # modelo de embeddings basado alternativo sentence-transformers/all-mpnet-base-v2
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}

embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

semantic_splitter = SemanticChunker(
   embeddings_model,
   breakpoint_threshold_type="percentile",
   breakpoint_threshold_amount=98
)

semantic_chunks = semantic_splitter.split_documents(all_docs)

print(f"Número de chunks con SemanticChunker: {len(semantic_chunks)}")
print("-" * 50)
print("Ejemplo de un chunk semántico:")
print(semantic_chunks[1].page_content)


/tmp/ipykernel_18283/3176677480.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


Número de chunks con SemanticChunker: 42
--------------------------------------------------
Ejemplo de un chunk semántico:
CSV: 2KVZ5BTI:1HCM7HFQ:KINEX1L4 URL de validación:https://www.tramita.gva.es/csv-front/index.faces?cadena=2KVZ5BTI:1HCM7HFQ:KINEX1L4


In [7]:
print(all_docs)

[Document(metadata={'producer': 'iText 2.1.7 by 1T3XT', 'creator': 'PyPDF', 'creationdate': '2025-04-11T07:11:52+02:00', 'moddate': '2025-04-11T11:26:49+02:00', 'source': 'data_antoni/20250415_NOTIFmODIF_firmado.pdf', 'total_pages': 1}, page_content='DIRECCIÓN TERRITORIAL \n \n \n \nCiudad Administrativa Nueve \nde Octubre \nTorre 4. Planta Baja \nCalle de la Democracia, 77 \n46018 VALENCIA \nTel: 012 \n \n \n \nPte/a. Asociación:  \nSOCIETAT UNIO MUSICAL ALBERIC \nPLAÇA CONSTITUCIÓ, 11 \n46260 Alberic \n \n \nASOCIACIONES \nEXPEDIENTE: 528 \nASUNTO: NOTIFICACIÓN MODIFICACIÓN DE ESTATUTOS DE LA \nASOCIACIÓN  "SOCIETAT UNIO MUSICAL ALBERIC" \n \n \nAdjunto se remite Resolución de modificación de los estatutos de la Asociación \n"SOCIETAT UNIO MUSICAL ALBERIC". \n \nSe remite asimismo una copia de los estatutos de la mencionada Asociación. \n \n \n \n \n \n \n \n \nCSV: 2KVZ5BTI:1HCM7HFQ:KINEX1L4 URL de validación:https://www.tramita.gva.es/csv-front/index.faces?cadena=2KVZ5BTI:1HCM7HFQ:

In [8]:
model_name = "intfloat/e5-large-v2"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': False}
embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

PARA SEMANTIC CHUNK




In [9]:
embeddings = embeddings_model.embed_documents(
    [split.page_content for split in semantic_chunks]
)
len(embeddings), len(embeddings[0])

(42, 1024)

In [10]:
from qdrant_client import QdrantClient, models


client = QdrantClient(host="localhost", port=6333)

collection_name = "dgt_documents_qdrant_memory_filter_fixed_2"


try:
    collection_info = client.get_collection(collection_name=collection_name)
    print(f"La colección '{collection_name}' ya existe.")
except Exception as e:
    print(f"La colección '{collection_name}' no existe. Creándola...")

    client.recreate_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=1024, 
            distance=models.Distance.COSINE
        )
    )
    print("Colección creada con éxito.")

collection_info = client.get_collection(collection_name=collection_name)
print("\nInformación de la colección:")
print(collection_info)




La colección 'dgt_documents_qdrant_memory_filter_fixed_2' no existe. Creándola...


/tmp/ipykernel_18283/720267713.py:15: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Colección creada con éxito.

Información de la colección:
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> vectors_count=None indexed_vectors_count=0 points_count=0 segments_count=8 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None), wal_config=WalConfig(wal_capaci

In [11]:
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaLLM
from langchain_openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

MODEL = "gpt-5-nano"
OLLAMA_MODEL = "gpt-oss:20b"
OLLAMA_MODEL_MISTRAL = "mistral:7b"
# llm_deeepseek = OllamaLLM(model="deepseek-r1", temperature=0.1, num_predict = 1024)
# llm_ollama = OllamaLLM(model=OLLAMA_MODEL_MISTRAL, temperature=0.1, num_predict = 1024)
llm_openai = ChatOpenAI(model=MODEL, temperature=0.1)

# llm_openai.chat.completion(messages=[{"role": "user", "content": "Hello, world!"}])

In [12]:
from pydantic import BaseModel, Field
from typing import Literal



TAXONOMIA_DOCUMENTAL = {
    "Documentos de la SUMA": "Documentos sobre la Sociedad Musical de Alberic en referencia a sus normas, actividades, gestión interna, documentacion.",
    "Manuales Técnicos y Procedimientos": "Manuales sobre el funcionamiento de las sociedad musicales, asi como recomendaciones y procedimientos para la organización de las juntas directivas.",
    "Inventarios y Activos SUMA": "Documentos relacionados con la gestión de inventarios, activos y recursos materiales de las sociedades musicales.",
    "Otros": "Documentos que no encajan claramente en ninguna de las anteriores."
}


CATEGORIAS_VALIDAS = list(TAXONOMIA_DOCUMENTAL.keys())

def clasificar_documento(nombre_archivo, texto_muestra):
    """
    Usa el LLM para decidir la categoría basándose en la descripción de la taxonomía.
    """
    # Convertimos el diccionario a texto para el prompt
    descripcion_categorias = "\n".join([f"- {k}: {v}" for k, v in TAXONOMIA_DOCUMENTAL.items()])
    
    prompt = f"""
    Eres un documentalista experto en Sociedades Musicales, especialmente en bandas sinfonicas.
    Tu misión es clasificar el siguiente documento en la categoría más adecuada.

    Detalles del documento:
    - Nombre: "{nombre_archivo}"
    - Inicio del texto: "{texto_muestra[:700]}..."

    Categorías disponibles y sus definiciones:
    {descripcion_categorias}


    Responde SOLAMENTE con el nombre exacto de la categoría.
    """
    
    try:
        response =llm_openai.invoke(prompt)
        categoria = response.content.strip().replace('"', '').replace("'", "").replace(".", "")
        

        for cat in CATEGORIAS_VALIDAS:
            if cat in categoria:
                return cat
        return "Otros"
    except:
        return "Otros"

# 3. Pre-calcular las categorías
archivo_a_categoria = {}
unique_files = set([doc.metadata['source'] for doc in semantic_chunks])

print("Clasificando documentos...")
for file_path in unique_files:
    filename = file_path.split('/')[-1]
    first_chunk = next((chunk.page_content for chunk in semantic_chunks if chunk.metadata['source'] == file_path), "")
    
    categoria_asignada = clasificar_documento(filename, first_chunk)
    archivo_a_categoria[file_path] = categoria_asignada
    print(f"📄 {filename[:40]}... -> {categoria_asignada}")

Clasificando documentos...
📄 03_2_Listado_Bienes.pdf... -> Inventarios y Activos SUMA
📄 Estatutos_SUM_Alberic_castellano.pdf... -> Documentos de la SUMA
📄 MANUAL DE DIRECTIVAS - def (v. enero 202... -> Manuales Técnicos y Procedimientos
📄 20250415_ESTATUTOS_DILIGENCIA.pdf... -> Documentos de la SUMA
📄 Estatutos_SUM_Alberic_castellano_.pdf... -> Documentos de la SUMA
📄 20250415_NOTIFmODIF_firmado.pdf... -> Documentos de la SUMA
📄 CIF_SUMA_Actualitzat.pdf... -> Otros


In [13]:
from qdrant_client import models
from tqdm.auto import tqdm
import uuid


batch_size = 100

print(f"Preparando para subir {len(semantic_chunks)} documentos en lotes de {batch_size}...\n")


valid_points_count = 0


for i in tqdm(range(0, len(semantic_chunks), batch_size)):
    batch_docs = semantic_chunks[i:i + batch_size]
    batch_embeddings = embeddings[i:i + batch_size]

    points_to_upsert = []
    for doc, emb in zip(batch_docs, batch_embeddings):
        content = doc.page_content
        source = doc.metadata.get("source", "")
    

        categoria = archivo_a_categoria.get(source, "Otros")


        if emb is not None:
            points_to_upsert.append(
                models.PointStruct(
                    id=str(uuid.uuid4()),
                    vector=emb,
                    payload={
                        "content": content,
                        "source": source,
                        "category": categoria, 
                        "page": doc.metadata.get("page", 0)
                    }
                )
            )
    
    if points_to_upsert: 
        client.upsert(
            collection_name=collection_name,
            points=points_to_upsert,
            wait=True
        )
        valid_points_count += len(points_to_upsert)


print("\n¡Subida de datos completada!")
print(f"Número de documentos válidos subidos: {valid_points_count}")

collection_info = client.get_collection(collection_name=collection_name)
print(f"\nVerificación: La colección ahora tiene {collection_info.points_count} puntos.")



Preparando para subir 42 documentos en lotes de 100...



  0%|          | 0/1 [00:00<?, ?it/s]


¡Subida de datos completada!
Número de documentos válidos subidos: 42

Verificación: La colección ahora tiene 42 puntos.


TEST LLM 

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


contextualize_q_system_prompt = """Dado un historial de chat y la última pregunta del usuario \
que podría hacer referencia al contexto en el historial de chat, formula una pregunta independiente \
que pueda entenderse sin el historial de chat. NO respondas a la pregunta, \
solo reformúlala si es necesario y, si no, devuélvela tal cual."""

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

qa_system_prompt = """Eres un asistente especializado en los documentos sobre sociedades musicales, especialmente de bandas sinfonicas. \
Utiliza los siguientes fragmentos de contexto recuperado para responder a la pregunta. \
Si no sabes la respuesta, di que no lo sabes. \
Menciona siempre de qué documentos has extraído la información explicitamente en el texto. \
Profundiza en la respuesta.

Contexto:
{context}"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

print("Creado correctamente los prompts de contextualización y QA.")

Creado correctamente los prompts de contextualización y QA.


In [15]:
from langchain_community.vectorstores import Qdrant



vectordb=Qdrant(
    client=client,
    collection_name=collection_name,
    embeddings=embeddings_model,
    content_payload_key="content"
)

retriever = vectordb.as_retriever(search_kwargs={"k": 5})

history_aware_retriever = create_history_aware_retriever(
    llm_openai, retriever, contextualize_q_prompt
)


question_answer_chain = create_stuff_documents_chain (llm_openai, qa_prompt)

rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)






/tmp/ipykernel_18283/2728287431.py:5: LangChainDeprecationWarning: The class `Qdrant` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-qdrant package and should be used instead. To use it run `pip install -U :class:`~langchain-qdrant` and import as `from :class:`~langchain_qdrant import Qdrant``.
  vectordb=Qdrant(


In [16]:
from qdrant_client.http import models as rest_models
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain



OPCIONES_CATEGORIAS = ["Todas"] + CATEGORIAS_VALIDAS

def build_qdrant_filter(category_name):
    """
    Construye el filtro técnico de Qdrant basándose en la selección del usuario.
    Si es 'Todas' o None, no aplica filtro.
    """
    if not category_name or category_name == "Todas":
        return None
    
    return rest_models.Filter(
        must=[
            rest_models.FieldCondition(
                key="category", 
                match=rest_models.MatchValue(value=category_name)
            )
        ]
    )

In [17]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",        
    history_messages_key="chat_history", 
    output_messages_key="answer",      
)

print("Sistema de memoria integrado. La cadena 'conversational_rag_chain' está lista.")

Sistema de memoria integrado. La cadena 'conversational_rag_chain' está lista.


In [18]:
def chat_con_filtros_manuales(message, history, selected_category):
    """
    Función de chat que recibe la categoría explícita desde la UI.
    """
    session_id = "demo_usuario"
    
    print(f"🔧 Filtro manual seleccionado: {selected_category}") 
    
    
    qdrant_filter = build_qdrant_filter(selected_category)
    
    # 2. Crear un retriever "al vuelo" con el filtro específico
    dynamic_retriever = vectordb.as_retriever(
        search_kwargs={
            "k": 5, 
            "filter": qdrant_filter 
        }
    )

    # 3. Reconstruir la cadena con el nuevo retriever
    # (Esto es necesario porque el filtro cambia la configuración del retriever)
    history_aware_retriever = create_history_aware_retriever(
        llm_openai, dynamic_retriever, contextualize_q_prompt
    )
    
    question_answer_chain = create_stuff_documents_chain(llm_openai, qa_prompt)
    
    dynamic_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)
    
    
    dynamic_conversational_chain = RunnableWithMessageHistory(
        dynamic_rag_chain,
        get_session_history, 
        input_messages_key="input",
        history_messages_key="chat_history",
        output_messages_key="answer",
    )

    
    full_response = ""
    for chunk in dynamic_conversational_chain.stream(
        {"input": message},
        config={"configurable": {"session_id": session_id}}
    ):
        if "answer" in chunk:
            full_response += chunk["answer"]
            yield full_response

In [19]:
# import gradio as gr

# filtro_dropdown = gr.Dropdown(
#     choices=OPCIONES_CATEGORIAS,
#     value="Todas", 
#     label="Filtrar por Categoría Documental",
#     info="Selecciona el tipo de documento donde quieres buscar la información."
# )

# iface = gr.ChatInterface(
#     fn=chat_con_filtros_manuales,  
#     additional_inputs=[filtro_dropdown],
#     title="Chatbot SUMA",
#     description="Pregunta sobre normativas, manuales o documentos relacionados con sociedades musicales. Selecciona una categoría específica para refinar la búsqueda.",
#     theme="soft"
# )

# iface.launch(debug=True)

In [20]:
import gradio as gr

# 1. Definir CSS personalizado (Fondo y transparencias)
custom_css = """
.gradio-container {
    background: url('/home/antonim/proyectos/proyextx/_37A5983.jpg') no-repeat center center fixed;
    background-size: cover;
}
/* Contenedor blanco semi-transparente para que se lea el chat */
.main-container {
    background-color: rgba(255, 255, 255, 0.95) !important; 
    border-radius: 15px;
    padding: 20px;
    margin-top: 20px;
    border: 1px solid #ddd;
}
/* Centrar imagen y textos */
.header-content {
    text-align: center;
    display: flex;
    flex-direction: column;
    align-items: center;
    justify-content: center;
}
"""

# 2. Configurar el tema (Colores)
tema_musical = gr.themes.Soft(
    primary_hue="indigo",  # Puedes cambiar a "red", "blue", etc.
    secondary_hue="slate",
).set(
    body_background_fill="transparent" # Importante para ver el fondo
)

# 3. Construir la Interfaz con Blocks
with gr.Blocks(theme=tema_musical, css=custom_css, title="Chatbot SUMA") as demo:
    
    # --- CABECERA ---
    with gr.Row():
        with gr.Column(elem_classes="header-content"):
            # Imagen del escudo (cambia la URL por tu archivo local si quieres)
            gr.Image(
                "/home/antonim/proyectos/proyextx/ESCUT BLANC I NEGRE AMB TEXT_lateral.png", 
                show_label=False, 
                show_download_button=False,
                container=False,
                height=100,
                width=100
            )
            gr.Markdown("""
            # 🎵 Asistente Virtual SUMA
            ### Sociedad Unió Musical de Alberic
            """)

    # --- CUERPO DEL CHAT ---
    with gr.Column(elem_classes="main-container"):
        
        # Filtro
        filtro_dropdown = gr.Dropdown(
            choices=OPCIONES_CATEGORIAS,
            value="Todas", 
            label="📂 Filtrar por Categoría",
            info="Selecciona el tipo de documento para acotar la búsqueda."
        )

        # ChatInterface (Versión compatible/simplificada)
        # Hemos quitado retry_btn, undo_btn, etc. para evitar el error.
        chat_interface = gr.ChatInterface(
            fn=chat_con_filtros_manuales,
            additional_inputs=[filtro_dropdown],
            examples=[
                ["¿Cuáles son los requisitos para ser socio?"], 
                ["Resumen del manual de procedimientos"],
                ["¿Cual es el domicilio social actualmente?"]
            ]
        )

# Lanzar
demo.launch(debug=True, share=False)

/home/antonim/miniconda3/envs/rag_env/lib/python3.12/site-packages/gradio/chat_interface.py:348: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


🔧 Filtro manual seleccionado: Documentos de la SUMA
🔧 Filtro manual seleccionado: Documentos de la SUMA
🔧 Filtro manual seleccionado: Manuales Técnicos y Procedimientos
🔧 Filtro manual seleccionado: Manuales Técnicos y Procedimientos
🔧 Filtro manual seleccionado: Inventarios y Activos SUMA
Keyboard interruption in main thread... closing server.


In [ ]:
!gradio --version

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Usage: gradio [OPTIONS] DEMO_PATH
Try 'gradio --help' for help.
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ No such option: --version                                                    │
╰──────────────────────────────────────────────────────────────────────────────╯
